# LoKI application

The application of `ess.instruments.loki` on `local()`, over the runs of the LoKI detector test at Larmor.
It needs the extra `[loki-app]`.

`FolderFiles` serves the files of one folder, here the download cache of esssans' tutorial data.
The first cell downloads the files that are missing.
The table gives each file's metadata. The `sample` column names the sample of a sample run and of its transmission run, and the batch pairs the two by it.
Run 60339 has no sample name, so the batch has no transmission run for it and does not reduce it.
Run 60393 is the can, and 60392 is its transmission run, also used as the empty beam.

In [ ]:
from pathlib import Path

import panel as pn
from ess.loki import data

from ess.apps import FolderFiles, local
from ess.instruments.loki.app import LokiApp
from ess.instruments.loki.bindings import BINDINGS

pn.extension('tabulator')

for download in (
    data.loki_tutorial_run_60392,
    data.loki_tutorial_background_run_60393,
    data.loki_tutorial_sample_run_60339,
    data.loki_tutorial_isis_polymer_sample_run,
    data.loki_tutorial_isis_polymer_transmission_run,
    data.loki_tutorial_agbeh_sample_run,
    data.loki_tutorial_agbeh_transmission_run,
    data.loki_tutorial_porous_silica_sample_run,
    data.loki_tutorial_porous_silica_transmission_run,
    data.loki_tutorial_deut_sds_sample_run,
    data.loki_tutorial_deut_sds_transmission_run,
    data.loki_tutorial_direct_beam_all_pixels,
):
    download()
mask_file = Path(data.loki_tutorial_mask_filenames()[0])
direct_beam_file = Path(data.loki_tutorial_direct_beam_all_pixels())

In [ ]:
def run(number: int, role: str, sample: str | None = None) -> dict[str, object]:
    row = {'file': f'{number}-2022-02-28_2215.nxs', 'run': number, 'role': role}
    return row | ({'sample': sample} if sample else {})


table = [
    run(60339, 'sample'),
    run(60395, 'sample', 'ISIS polymer'),
    run(60394, 'transmission', 'ISIS polymer'),
    run(60387, 'sample', 'AgBeh'),
    run(60386, 'transmission', 'AgBeh'),
    run(60385, 'sample', 'porous silica'),
    run(60384, 'transmission', 'porous silica'),
    run(60389, 'sample', 'deut-SDS'),
    run(60388, 'transmission', 'deut-SDS'),
    run(60393, 'background'),
    run(60392, 'transmission'),
    {'file': mask_file.name, 'kind': 'mask'},
    {'file': direct_beam_file.name, 'kind': 'calibration', 'role': 'direct_beam'},
]
files = FolderFiles(mask_file.parent, table)
client = local(proposal='loki-test', files=files, bind=BINDINGS)

## The order of use

1. **Pixel masks**: pick the form *Pixel masks* above the tabs, give run 60339 and the mask file, and submit. In the tab *Outputs*, select the image and pick the view *Masks on image*.
2. **Beam centre**: pick the form *Beam centre*, give run 60339 and the masks as *From a record*, and submit. The view *Centre on image* marks the centre.
3. **Batch**: in the tab *Batch*, give the settings of all runs: the can, its transmission run, the empty beam, the direct beam, the masks and the beam centre *From a record*, and the bins. **Reduce all** reduces every row that has a transmission run, and the plot shows their I(Q).
4. To change one run, select its row, press **Edit**, change a value in the form at the bottom, and press **Submit**. The new result replaces the row's result in the table and in the plot.

In [ ]:
app = LokiApp(client)
app.panel()